# Train UCL caveat at the paper's scale on NHTS 2017 (split 1)
Runtime > Change runtime type > **T4**, then **Runtime > Run all**.
Generated by colab-factory. If a cell fails, send that cell's text output to Claude.

In [ ]:
import sys, subprocess
print(sys.version)
print(subprocess.run('nvidia-smi -L || echo NO-GPU', shell=True, capture_output=True, text=True).stdout)

In [ ]:
!pip install -q "pynhts==0.2.1" "pyarrow"

In [ ]:
%%writefile /content/pb/diary.py
"""Discretised person-day diaries from NHTS trips (pilot).

A diary is a ``(n_persons, 288)`` uint8 matrix of 5-minute slots covering the NHTS travel day
(04:00 to 04:00). States: 0 home, 1 work, 2 school/care, 3 maintenance (errands, escort,
health), 4 leisure (meals, social, recreation, religious, volunteer), 5 other, 6 travelling.
"""

from __future__ import annotations

import numpy as np
import pandas as pd

N_SLOTS = 288
SLOT_MIN = 5
DAY_START_MIN = 240  # 04:00
STATES = ["home", "work", "school", "maint", "leisure", "other", "travel"]
H, W, S, M, L, O, T = range(7)

_WHY = {
    "01": H, "02": H, "03": W, "04": W, "08": S, "09": S, "10": S,
    "06": M, "11": M, "12": M, "14": M, "18": M,
    "13": L, "15": L, "16": L, "17": L, "19": L, "05": L,
    "07": O, "97": O,
}


def _slot(hhmm: pd.Series) -> np.ndarray:
    """Slot index of an ``hhmm`` string; times before 04:00 belong to the next calendar day."""
    v = pd.to_numeric(hhmm, errors="coerce").to_numpy(dtype=np.float64)
    minutes = (v // 100) * 60 + (v % 100)
    minutes = np.where(minutes < DAY_START_MIN, minutes + 1440, minutes)
    return np.asarray((minutes - DAY_START_MIN) // SLOT_MIN)


def person_covariates(persons: pd.DataFrame) -> pd.DataFrame:
    """Conditioning variables used by every model (age band, worker, vehicles, weekend)."""
    age = persons["R_AGE_IMP"].to_numpy(dtype=np.float64)
    band = np.digitize(age, [5, 18, 25, 45, 65])  # 0:<5 1:5-17 2:18-24 3:25-44 4:45-64 5:65+
    worker = (persons["WORKER"] == "01").fillna(False).to_numpy().astype(int)
    veh = np.minimum(persons["HHVEHCNT"].to_numpy(dtype=np.float64), 2).astype(int)
    weekend = persons["TRAVDAY"].isin(["01", "07"]).to_numpy().astype(int)
    return pd.DataFrame({"age": band, "worker": worker, "veh": veh, "weekend": weekend},
                        index=persons.index)


def build_diaries(persons: pd.DataFrame, trips: pd.DataFrame) -> tuple[np.ndarray, dict[str, int]]:
    """Diary matrix for ``persons`` (row order preserved) from their ``trips``.

    Returns the matrix and a small report (trips used / skipped).
    """
    n = len(persons)
    diary = np.zeros((n, N_SLOTS), dtype=np.uint8)
    key = pd.MultiIndex.from_frame(persons[["HOUSEID", "PERSONID"]])
    t = trips.copy()
    t["_p"] = key.get_indexer(pd.MultiIndex.from_frame(t[["HOUSEID", "PERSONID"]]))
    t["_n"] = pd.to_numeric(t["TDTRPNUM"])
    t["_s"], t["_e"] = _slot(t["STRTTIME"]), _slot(t["ENDTIME"])
    wrap = t["_e"] < t["_s"]
    t.loc[wrap, "_e"] = t.loc[wrap, "_e"] + 1440 // SLOT_MIN
    bad = (t["_p"] < 0) | t["_s"].isna() | t["_e"].isna()
    report = {"trips": len(t), "skipped": int(bad.sum())}
    t = t[~bad].sort_values(["_p", "_n"], kind="stable")
    p = t["_p"].to_numpy()
    s = np.clip(t["_s"].to_numpy(dtype=np.int64), 0, N_SLOTS - 1)
    e = np.clip(t["_e"].to_numpy(dtype=np.int64), 0, N_SLOTS - 1)
    to = t["WHYTO"].map(_WHY).fillna(O).to_numpy(dtype=np.uint8)
    fr = t["WHYFROM"].map(_WHY).fillna(H).to_numpy(dtype=np.uint8)
    nxt = np.append(s[1:], N_SLOTS)
    nxt = np.where(np.append(p[1:] == p[:-1], False), nxt, N_SLOTS)
    first = np.append(True, p[1:] != p[:-1])
    for i in range(len(p)):
        row = diary[p[i]]
        if first[i]:
            row[: s[i]] = fr[i]
        row[s[i]: e[i] + 1] = T
        if e[i] + 1 < nxt[i]:
            row[e[i] + 1: nxt[i]] = to[i]
    return diary, report


def person_stats(diary: np.ndarray) -> pd.DataFrame:
    """Per-person outcomes whose weighted means are the benchmark targets."""
    trav = diary == T
    starts = trav[:, 1:] & ~trav[:, :-1]
    trips = starts.sum(axis=1) + trav[:, 0]
    minutes = {f"min_{name}": (diary == k).sum(axis=1) * SLOT_MIN for k, name in enumerate(STATES)}
    first_dep = np.where(trav.any(axis=1), trav.argmax(axis=1) * SLOT_MIN, np.nan)
    at = lambda hhmm: (hhmm - DAY_START_MIN) // SLOT_MIN  # noqa: E731
    out = pd.DataFrame({
        "any_trip": (trips > 0).astype(float),
        "trips": trips.astype(float),
        "trips_3plus": (trips >= 3).astype(float),
        **{k: v.astype(float) for k, v in minutes.items()},
        "has_work": (diary == W).any(axis=1).astype(float),
        "first_departure_min": first_dep,
        "work_at_10": (diary[:, at(600)] == W).astype(float),
        "travel_at_08": (diary[:, at(480)] == T).astype(float),
        "travel_at_17": (diary[:, at(1020)] == T).astype(float),
        "home_at_23": (diary[:, at(1380)] == H).astype(float),
    })
    return out


In [ ]:
%%writefile /content/pb/schedule.py
"""Activity-schedule representation (no travel state) shared by every model in the sprint.

A schedule is a table ``pid, act, start, end`` (minutes since 04:00, 0..1440): a sequence of
activities where the time spent travelling to an activity belongs to that activity, the format
`caveat` (UCL) uses. Every model's output is brought to this form before it is scored, so models
that can and cannot produce a travel state are compared on the same footing.
"""

from __future__ import annotations

import numpy as np
import pandas as pd

from diary import N_SLOTS, SLOT_MIN, STATES

ACTS = ["home", "work", "school", "maint", "leisure", "other"]  # STATES without "travel"
T = 6


def diary_to_schedule(diary: np.ndarray, pids: np.ndarray | None = None) -> pd.DataFrame:
    """Convert a ``(n, 288)`` diary matrix to a schedule table (a new activity starts at each
    departure; travel time is absorbed into the destination activity)."""
    n = len(diary)
    pids = np.arange(n) if pids is None else np.asarray(pids)
    travel = diary == T
    # destination state for each slot: next non-travel state; fall back to the last one seen
    nxt = np.full(n, 255, dtype=np.uint8)
    dest = np.empty_like(diary)
    for t in range(N_SLOTS - 1, -1, -1):
        nxt = np.where(travel[:, t], nxt, diary[:, t])
        dest[:, t] = nxt
    prv = np.zeros(n, dtype=np.uint8)
    for t in range(N_SLOTS):
        prv = np.where(travel[:, t], prv, diary[:, t])
        dest[:, t] = np.where(dest[:, t] == 255, prv, dest[:, t])
    begin = np.zeros_like(travel)
    begin[:, 0] = True
    begin[:, 1:] = travel[:, 1:] & ~travel[:, :-1]
    rows, cols = np.nonzero(begin)  # row-major => sorted by row then column
    nxt_col = np.append(cols[1:], N_SLOTS)
    last = np.append(rows[1:] != rows[:-1], True)
    end_col = np.where(last, N_SLOTS, nxt_col)
    act = dest[rows, cols]
    out = pd.DataFrame({"pid": pids[rows], "act": np.array(ACTS, dtype=object)[act],
                        "start": cols * SLOT_MIN, "end": end_col * SLOT_MIN})
    return out


def schedule_stats(sched: pd.DataFrame, pids: np.ndarray) -> pd.DataFrame:
    """Person-level outcomes of a schedule table, one row per ``pids`` entry (in that order)."""
    s = sched.sort_values(["pid", "start"], kind="stable").reset_index(drop=True)
    s["dur"] = s["end"] - s["start"]
    g = s.groupby("pid")
    n_acts = g.size().reindex(pids).fillna(0).to_numpy()
    out = pd.DataFrame(index=np.arange(len(pids)))
    trips = np.maximum(n_acts - 1, 0)
    out["any_trip"] = (trips > 0).astype(float)
    out["trips"] = trips
    out["trips_3plus"] = (trips >= 3).astype(float)
    mins = s.pivot_table(index="pid", columns="act", values="dur", aggfunc="sum",
                         fill_value=0).reindex(pids).fillna(0)
    for a in ACTS:
        out[f"min_{a}"] = mins[a].to_numpy() if a in mins else 0.0
    out["has_work"] = (out["min_work"] > 0).astype(float)
    first = g.head(1).set_index("pid").reindex(pids)
    out["first_departure_min"] = np.where(trips > 0, first["end"].to_numpy(), np.nan)
    last = g.tail(1).set_index("pid").reindex(pids)
    out["ends_at_home"] = (last["act"].to_numpy() == "home").astype(float)
    for name, t in (("work_at_10", 360), ("home_at_23", 1140)):
        hit = s[(s["start"] <= t) & (s["end"] > t)]
        at = hit.drop_duplicates("pid").set_index("pid")["act"].reindex(pids)
        out[name] = (at.to_numpy() == ("work" if name.startswith("work") else "home")).astype(float)
    w = s[s["act"] == "work"].drop_duplicates("pid").set_index("pid")["start"].reindex(pids)
    out["work_start_min"] = w.to_numpy(dtype=float)
    return out


In [ ]:
%%writefile /content/export_inputs.py

import hashlib, os, sys
import numpy as np, pandas as pd
sys.path.insert(0, "/content/pb")
import pynhts
from diary import build_diaries, person_covariates
from schedule import diary_to_schedule

SPLIT = int(os.environ.get("SPLIT", "0"))  # seed 1000 + split, same as common.split in the local pipeline
# sha256 of (train pids, test pids, train_schedules.csv with LF line endings; Windows writes CRLF) from the local export_caveat.py output
EXPECTED = {
    0: ("501683a613223f4ab2ddc3efdac75fae408f984d8d00bc67b11474c4c797bd19",
        "b138f71999e2d1e9e26f732d138e31dcc0f266a013ce5efc52f417b78aab4dad",
        "371fba958e526c69ddf3b4fabfc6899d6cee6ade58540d344d58b5348bb84f88"),
    1: ("caf6c8fbe56f9b3345b03398302d9d11e6c0052e675831814fdcf5fc0c810ed2",
        "66e4c05c5f46c052a8d7306b086bba695f34b35e867aef5c8e2018c2f1b34917",
        "0ea21753dadb45577d5ac1420d93c3abad1a07a6545bfa6401ec53b0daa037dd"),
    2: ("ec76e2d698b5536188d8c3e216de0e96591b003cf8d6fc1417a5a993bf8079ce",
        "d5fd27efc8a0cf8c4e1b4a1dc4aff835db0f9ada11b019bfc391c3fac33a4044",
        "d2968a10684d6439a2a7e39d1e3c5f98bdb9768789834a5d118fe50fcc3f92e0"),
    3: ("0a125b87d944ad96cb2bfc211e7464471c8c35d9adb3ec6270946567dede92ef",
        "3820f5fcada52e76d0d51d878ea6a0fc357ad6dd28b2369fbd971ca95a327154",
        "623e0a61c95787e90f882b23ce283a8f34dfe62bb2b0ddf039cb46e1d60319c6"),
    4: ("229a7e25d7fe19a99959977bad5107093997d3edc470d29b28a1d68d459fc1d1",
        "0a0312f0755b511c9bc2479c1abc40546f69eb2d6384e130526dbbecd42d98dd",
        "fb029b20242c8c512a7342f44ae6317481d691a48c9e31d504ad11e5af3c6fb9"),
}
CRLF, LF = bytes([13, 10]), bytes([10])
d = pynhts.load(2017)
persons = d.persons(["R_AGE_IMP", "WORKER", "HHVEHCNT", "TRAVDAY"])
trips = d.trips(["TDTRPNUM", "STRTTIME", "ENDTIME", "WHYTO", "WHYFROM"], keys=True)
diary, rep = build_diaries(persons, trips); print(rep)
cov = person_covariates(persons)
hh = persons["HOUSEID"].to_numpy(); uh = np.unique(hh)
rng = np.random.default_rng(1000 + SPLIT)
te = np.isin(hh, list(set(rng.choice(uh, size=len(uh) // 2, replace=False)))); tr = ~te
sched = diary_to_schedule(diary)
n = sched.groupby("pid").size(); keep = n.index[n <= 16]
def attrs(mask):
    c = cov[mask]
    return pd.DataFrame({"pid": np.flatnonzero(mask), "age": c["age"].astype(str).to_numpy(),
        "worker": c["worker"].astype(str).to_numpy(), "veh": c["veh"].astype(str).to_numpy(),
        "weekend": c["weekend"].astype(str).to_numpy()})
tr_pids = np.intersect1d(np.flatnonzero(tr), keep)
out = "/content/caveat_run"
os.makedirs(out, exist_ok=True)
sched[sched.pid.isin(tr_pids)].to_csv(out + "/train_schedules.csv", index=False)
attrs(np.isin(np.arange(len(tr)), tr_pids)).to_csv(out + "/train_attributes.csv", index=False)
attrs(te).to_csv(out + "/test_attributes.csv", index=False)
h = lambda p: hashlib.sha256(",".join(map(str, pd.read_csv(p)["pid"])).encode()).hexdigest()
got = (h(out + "/train_attributes.csv"), h(out + "/test_attributes.csv"),
       hashlib.sha256(open(out + "/train_schedules.csv", "rb").read().replace(CRLF, LF)).hexdigest())
print("split", SPLIT, "hashes:", got)
if SPLIT in EXPECTED:
    ok = tuple(a == b for a, b in zip(got, EXPECTED[SPLIT]))
    print("matches local export (train pids, test pids, train schedules):", ok)
    assert all(ok[:2]), "household split differs from the local one: stop and tell Claude"
    if not ok[2]:
        print("WARNING: schedule bytes differ from local (formatting?); pids match, continuing")
else:
    print("no local hashes registered for this split: cannot verify against the laptop")


In [ ]:
%%writefile /content/caveat_run/run_full.yaml
logging_params:
  log_dir: "/content/caveat_run/logs"
  name: "nhts_split1_full"
schedules_path: "/content/caveat_run/train_schedules.csv"
attributes_path: "/content/caveat_run/train_attributes.csv"
synthetic_attributes_path: "/content/caveat_run/test_attributes.csv"
labels_encoder:
  name: "tokens"
  labels:
    age: nominal
    worker: nominal
    veh: nominal
    weekend: nominal
encoder_params:
  name: "continuous"
  max_length: 16
  norm_duration: 1440
  jitter: 0
  fix_durations: stretch
  trim_eos: True
  weighting: unit
  joint_weighting: unit
model_params:
  name: "CVAEContLSTMCP"
  hidden_n: 4
  hidden_size: 256
  labels_hidden_size: 64
  latent_dim: 6
  dropout: 0.0
  teacher_forcing_ratio: 0.5
  hidden_conditionality: add
  encoder_conditionality: none
  latent_conditionality: add
  decoder_conditionality: none
loader_params:
  train_batch_size: 1024
  val_batch_size: 1024
  num_workers: 2
experiment_params:
  LR: 0.001
  weight_decay: 0.01
  scheduler_gamma: 0.98
  kld_weight: 0.01
  duration_loss_weight: 200
trainer_params:
  patience: 50
  max_epochs: 300
  min_epochs: 100
seed: 1234


In [ ]:
![ -e /content/caveat_run/train_schedules.csv ] && echo 'already done, skipping' || ( SPLIT=1 python /content/export_inputs.py )

## Isolated environment (STOP and report if this cell ends with an ImportError)

In [ ]:
!pip install -q uv
!uv venv /content/venv --python 3.12
!rm -rf /content/caveat && git clone -q https://github.com/big-ucl/caveat.git /content/caveat
!cd /content/caveat && git checkout -q f4a231e79c24a60f0db9a003ea1ac1fd8eeaa194 && git log -1 --format=%H
!uv pip install --python /content/venv/bin/python "torch" "torchvision" "scipy<=1.13" "numpy<2" "pandas<3" "pytorch-lightning" "pot" "click" "tabulate" "matplotlib" "tensorboard" "optuna" "optuna-integration" "pyproj" "xlrd" "acteval" "torchmetrics" "scikit-learn" "seaborn" "networkx" "pyyaml" "tqdm"
!uv pip install --python /content/venv/bin/python --no-deps "cml-pam"
!uv pip install --python /content/venv/bin/python --no-deps -e /content/caveat
!cd /tmp && MPLBACKEND=Agg /content/venv/bin/python -c "import torch, torchvision, torchmetrics, sklearn, seaborn, networkx, yaml, tqdm, pam, caveat.cli; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
!SPLIT=1 MPLBACKEND=Agg PYTHONUTF8=1 stdbuf -oL bash -c 'cd /content/caveat && /content/venv/bin/caveat run /content/caveat_run/run_full.yaml' 2>&1 | tee /content/train.log | tr '\r' '\n' | grep --line-buffered -E 'Epoch [0-9]+: 100%|Generation took|Results saved|rror|Traceback'
!tail -n 25 /content/train.log | tr '\r' '\n' | tail -n 25

In [ ]:
import glob, os, subprocess
found = subprocess.run("cd /content/caveat_run && find . \\( -path './logs/*/synthetic_schedules.csv' -o -path './logs/*/synthetic_attributes.csv' -o -path './logs/*/hparams.yaml' -o -path './logs/*/timer.csv' \\)", shell=True, capture_output=True, text=True).stdout.split()
print('output files:', found)
if not found:
    print(open('/content/train.log').read()[-3000:])
    raise SystemExit('run produced no outputs: send the text above to Claude')
!cd /content/caveat_run && find . \( -path './logs/*/synthetic_schedules.csv' -o -path './logs/*/synthetic_attributes.csv' -o -path './logs/*/hparams.yaml' -o -path './logs/*/timer.csv' \) | zip -@ /content/caveat_outputs_split1.zip
from google.colab import files
files.download('/content/caveat_outputs_split1.zip')